<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%); padding: 2.5rem 2rem; border-radius: 12px; margin-bottom: 1.5rem;">
  <h1 style="color: #ffffff; margin: 0 0 0.5rem 0; font-size: 1.8rem;">🔧 Lab 03a: Add Financial Data Tools to Your Advisor Agent</h1>
  <p style="color: #a0cfee; margin: 0; font-size: 1.05rem;">Zava Bank Workshop — Microsoft Foundry Agent Observability</p>
</div>

<div style="background-color: #e8f4fd; border-left: 4px solid #0078d4; padding: 1.2rem 1.5rem; border-radius: 6px; margin-bottom: 1.5rem;">

## What We Learn

In this lab we give our advisor agent **real data access** through `FunctionTool` objects backed by CSV-loaded DataFrames. The agent gains three capabilities:

| Tool | Purpose |
|---|---|
| `search_portfolio` | Look up client holdings by name, ID, ticker, or sector |
| `assess_risk` | Retrieve risk tolerance, beta, Sharpe ratio, and concentration metrics |
| `get_market_data` | Pull current indices, sector performance, and economic indicators |

With these tools the agent transforms from a general financial chatbot into a **data-driven advisory assistant** that can answer real questions about real clients.

</div>

<div style="background-color: #fff8e1; border-left: 4px solid #ffb300; padding: 1.2rem 1.5rem; border-radius: 6px; margin-bottom: 1.5rem;">

## 🏦 The Zava Bank Story

In Lab 02 we built a concierge agent that could *talk* about finance — it had strong instructions, a compliance-aware persona, and polished conversational skills. But it had **no actual data**. Ask it about Margaret Chen's portfolio and it would either hallucinate or politely decline.

Now we connect the agent to Zava Bank's systems:

- **Client portfolios** — holdings, cost basis, current prices, sectors
- **Risk metrics** — beta, Sharpe ratio, drawdown, concentration, sector exposure
- **Market data** — broad indices, sector performance, economic indicators

This is the difference between a chatbot and a useful tool. After this lab the agent can prepare a client briefing, flag risk concerns, and cite actual numbers — the kind of work a junior analyst would do before a meeting.

</div>

---
## 1 · Environment Setup

In [ ]:
import os, json, pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, Tool, FunctionTool

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint   = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
credential = DefaultAzureCredential()

project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client  = project_client.get_openai_client()

print(f"Project endpoint : {endpoint}")
print(f"Model deployment : {model_name}")

---
## 2 · Load Zava Bank Data

In [ ]:
DATA_DIR = os.path.join(
    os.path.dirname(os.path.abspath(".")), "..", "data", "zava-bank"
)

portfolios_df = pd.read_csv(os.path.join(DATA_DIR, "client_portfolios.csv"))
market_df     = pd.read_csv(os.path.join(DATA_DIR, "market_data.csv"))
risk_df       = pd.read_csv(os.path.join(DATA_DIR, "risk_metrics.csv"))

print(
    f"Loaded {len(portfolios_df)} portfolio positions, "
    f"{len(market_df)} market indicators, "
    f"{len(risk_df)} client risk profiles"
)

In [ ]:
# Quick look at the data the agent will query
display(portfolios_df.head(3))
display(risk_df.head(3))
display(market_df.head(3))

---
## 3 · Define Tool Functions

Each function accepts optional filter parameters, queries the relevant DataFrame, and returns JSON. The agent will call these functions by name during a conversation.

### 3.1 · `search_portfolio`

In [ ]:
def search_portfolio(
    client_name: str = None,
    client_id: str = None,
    ticker: str = None,
    sector: str = None,
) -> str:
    """Search client portfolio holdings based on criteria."""
    results = portfolios_df.copy()
    if client_name:
        results = results[
            results["client_name"].str.lower().str.contains(client_name.lower())
        ]
    if client_id:
        results = results[
            results["client_id"].str.upper() == client_id.upper()
        ]
    if ticker:
        results = results[
            results["ticker"].str.upper() == ticker.upper()
        ]
    if sector:
        results = results[
            results["sector"].str.lower().str.contains(sector.lower())
        ]
    if results.empty:
        return json.dumps({"message": "No portfolio holdings found matching your criteria."})
    return results.to_json(orient="records")

### 3.2 · `assess_risk`

In [ ]:
def assess_risk(
    client_name: str = None,
    client_id: str = None,
    risk_tolerance: str = None,
) -> str:
    """Retrieve risk assessment metrics for a client portfolio."""
    results = risk_df.copy()
    if client_name:
        results = results[
            results["client_name"].str.lower().str.contains(client_name.lower())
        ]
    if client_id:
        results = results[
            results["client_id"].str.upper() == client_id.upper()
        ]
    if risk_tolerance:
        results = results[
            results["risk_tolerance"].str.lower() == risk_tolerance.lower()
        ]
    if results.empty:
        return json.dumps({"message": "No risk data found matching your criteria."})
    return results.to_json(orient="records")

### 3.3 · `get_market_data`

In [ ]:
def get_market_data(
    category: str = None,
    name: str = None,
) -> str:
    """Get current market data including indices, sector performance, and economic indicators."""
    results = market_df.copy()
    if category:
        results = results[
            results["category"].str.lower().str.contains(category.lower())
        ]
    if name:
        results = results[
            results["name"].str.lower().str.contains(name.lower())
        ]
    if results.empty:
        return json.dumps({"message": "No market data found matching your criteria."})
    return results.to_json(orient="records")

### Quick smoke test — make sure the functions return data before we hand them to the agent.

In [ ]:
print("Portfolio search →", search_portfolio(client_name="Margaret Chen")[:120], "...")
print("Risk assessment  →", assess_risk(client_id="ZB-CL-002")[:120], "...")
print("Market data      →", get_market_data(category="Index")[:120], "...")

---
## 4 · Define FunctionTool Schemas

Each `FunctionTool` wraps a JSON Schema that tells the model what parameters the function accepts. The model uses these schemas to decide which tool to call and what arguments to pass.

In [ ]:
portfolio_tool = FunctionTool(
    name="search_portfolio",
    description="Search client portfolio holdings by client name, client ID, stock ticker, or sector.",
    parameters={
        "type": "object",
        "properties": {
            "client_name": {
                "type": "string",
                "description": "Full or partial client name (e.g. 'Margaret Chen', 'Apex').",
            },
            "client_id": {
                "type": "string",
                "description": "Zava Bank client identifier (e.g. 'ZB-CL-001').",
            },
            "ticker": {
                "type": "string",
                "description": "Stock ticker symbol (e.g. 'ZVA', 'MTK').",
            },
            "sector": {
                "type": "string",
                "description": "Market sector (e.g. 'Technology', 'Healthcare').",
            },
        },
        "required": [],
    },
)

risk_tool = FunctionTool(
    name="assess_risk",
    description="Retrieve risk assessment metrics for a client portfolio, including beta, Sharpe ratio, drawdown, and concentration.",
    parameters={
        "type": "object",
        "properties": {
            "client_name": {
                "type": "string",
                "description": "Full or partial client name.",
            },
            "client_id": {
                "type": "string",
                "description": "Zava Bank client identifier.",
            },
            "risk_tolerance": {
                "type": "string",
                "description": "Filter by risk tolerance level.",
                "enum": [
                    "Conservative",
                    "Moderate-Conservative",
                    "Moderate",
                    "Moderate-Aggressive",
                    "Aggressive",
                ],
            },
        },
        "required": [],
    },
)

market_tool = FunctionTool(
    name="get_market_data",
    description="Get current market data including broad indices, sector performance, and economic indicators.",
    parameters={
        "type": "object",
        "properties": {
            "category": {
                "type": "string",
                "description": "Category of market data to retrieve.",
                "enum": ["Index", "Sector Performance", "Economic Indicator"],
            },
            "name": {
                "type": "string",
                "description": "Full or partial indicator name (e.g. 'Tech Composite', 'CPI').",
            },
        },
        "required": [],
    },
)

print("Defined 3 FunctionTool schemas: search_portfolio, assess_risk, get_market_data")

---
## 5 · Create the Advisor Agent (with Tools)

In [ ]:
ADVISOR_INSTRUCTIONS = """
You are a senior financial advisor at Zava Bank, a premium wealth-management firm.
You serve high-net-worth individuals and institutional clients with data-driven,
compliance-aware investment guidance.

## Your tools
You have three tools that connect you to Zava Bank's systems:

1. **search_portfolio** — Look up client holdings. Use this whenever a user asks
   about a client's positions, allocations, or specific stocks they own.
2. **assess_risk** — Retrieve risk metrics (beta, Sharpe, drawdown, concentration).
   Use this when discussing risk tolerance, portfolio risk, or suitability.
3. **get_market_data** — Pull current indices, sector performance, and economic
   indicators. Use this to provide market context for recommendations.

Always call the relevant tools before answering data questions — never guess at
portfolio contents or market figures.

## Communication style
- Address clients and colleagues professionally
- Lead with the most important insight
- Cite specific numbers from the data you retrieve
- When portfolio data shows unrealized gains/losses, calculate and mention them
- Flag any concentration risk or misalignment with stated risk tolerance

## Compliance rules
- Never guarantee future returns or make promissory statements
- Include a brief disclaimer when giving investment-related opinions
- Recommend professional consultation for tax or legal matters
- If you don't have data to answer a question, say so — do not fabricate
""".strip()

print(ADVISOR_INSTRUCTIONS[:200], "...")

In [ ]:
agent = project_client.agents.create_version(
    agent_name="zava-bank-advisor-with-tools",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=ADVISOR_INSTRUCTIONS,
        tools=[
            Tool(function=portfolio_tool),
            Tool(function=risk_tool),
            Tool(function=market_tool),
        ],
    ),
)

print(f"Agent created: {agent.name}  (version {agent.version})")

---
## 6 · Tool-Call Handler

When the model decides to call a tool, the response contains `function_call` items instead of text. We need to:
1. Parse the arguments the model chose
2. Call the local Python function
3. Return the result so the model can incorporate it into its answer

In [ ]:
from openai.types.responses.response_input_param import FunctionCallOutput


def handle_tool_calls(response):
    """Execute every function_call in the response and return results."""
    tool_map = {
        "search_portfolio": search_portfolio,
        "assess_risk": assess_risk,
        "get_market_data": get_market_data,
    }

    function_results = []
    for item in response.output:
        if item.type == "function_call":
            func = tool_map.get(item.name)
            if func:
                args = json.loads(item.arguments)
                print(f"  🔧 Calling {item.name}({args})")
                result = func(**args)
                function_results.append(
                    FunctionCallOutput(
                        type="function_call_output",
                        call_id=item.call_id,
                        output=result,
                    )
                )
            else:
                print(f"  ⚠️  Unknown tool: {item.name}")
    return function_results

---
## 7 · Agentic Loop

The loop sends a user message, checks if the model wants to call tools, executes them, feeds results back, and repeats until the model produces a final text response.

In [ ]:
def agentic_loop(user_message: str, *, max_turns: int = 10) -> str:
    """Run a full tool-augmented conversation and return the final text."""
    print(f"\n{'='*60}")
    print(f"USER: {user_message}")
    print(f"{'='*60}")

    # Initial request
    response = openai_client.responses.create(
        model=model_name,
        instructions=ADVISOR_INSTRUCTIONS,
        input=user_message,
        tools=[
            {"type": "function", "function": portfolio_tool.as_dict()},
            {"type": "function", "function": risk_tool.as_dict()},
            {"type": "function", "function": market_tool.as_dict()},
        ],
    )

    turn = 0
    while turn < max_turns:
        # Check if the model wants to call tools
        tool_calls = [item for item in response.output if item.type == "function_call"]
        if not tool_calls:
            break

        print(f"\n--- Tool turn {turn + 1} ({len(tool_calls)} call(s)) ---")
        function_results = handle_tool_calls(response)

        # Feed tool results back to the model
        response = openai_client.responses.create(
            model=model_name,
            instructions=ADVISOR_INSTRUCTIONS,
            input=[
                {"type": "message", "role": "user", "content": user_message},
                *[{"type": "function_call", "name": tc.name, "call_id": tc.call_id, "arguments": tc.arguments} for tc in tool_calls],
                *[fr.model_dump() if hasattr(fr, 'model_dump') else fr for fr in function_results],
            ],
            tools=[
                {"type": "function", "function": portfolio_tool.as_dict()},
                {"type": "function", "function": risk_tool.as_dict()},
                {"type": "function", "function": market_tool.as_dict()},
            ],
        )
        turn += 1

    # Extract final text
    final_text = "\n".join(
        item.text for item in response.output if hasattr(item, "text")
    )
    print(f"\nADVISOR:\n{final_text}")
    return final_text

---
## 8 · Test the Agent

Three queries that exercise different tool combinations.

### Test 1 — Single-tool: Portfolio lookup

In [ ]:
_ = agentic_loop(
    "Pull up Margaret Chen's complete portfolio and tell me about her current positions."
)

### Test 2 — Single-tool: Risk assessment

In [ ]:
_ = agentic_loop(
    "What's the risk profile for Jason Rivera? Are there any concerns I should flag?"
)

### Test 3 — Multi-tool: Full client briefing

This is where it gets interesting. The agent should call all three tools to build a comprehensive briefing.

In [ ]:
_ = agentic_loop(
    "Give me a full briefing for my meeting with Apex Capital Partners "
    "— portfolio, risk, and relevant market context."
)

---
## 9 · Cleanup

In [ ]:
# Delete the agent version we created
project_client.agents.delete_version(
    agent_name=agent.name,
    version=agent.version,
)
print(f"Deleted agent version: {agent.name} v{agent.version}")

# Close clients
openai_client.close()
project_client.close()
print("Clients closed.")

---

<div style="background-color: #e6f9e6; border-left: 4px solid #2e7d32; padding: 1.2rem 1.5rem; border-radius: 6px; margin-bottom: 1rem;">

## ✅ What We Accomplished

- Loaded Zava Bank's client portfolios, risk metrics, and market data from CSV
- Defined three Python functions that query DataFrames and return JSON
- Wrapped those functions in `FunctionTool` schemas the model can understand
- Built a tool-call handler that maps model requests to local function execution
- Implemented an agentic loop that handles multi-turn tool calling
- Tested single-tool and multi-tool queries against real client data

</div>

<div style="background-color: #e8f4fd; border-left: 4px solid #0078d4; padding: 1.2rem 1.5rem; border-radius: 6px; margin-bottom: 1rem;">

## 💡 Key Takeaway

Function tools are what turn an LLM from a text generator into a working system. The model decides *when* to call a tool, *which* tool to use, and *what arguments* to pass — all based on the user's natural-language request. Our job is to give it well-defined schemas and reliable functions.

</div>

<div style="background-color: #f3e5f5; border-left: 4px solid #7b1fa2; padding: 1.2rem 1.5rem; border-radius: 6px;">

## ➡️ Next Steps

In **Lab 03b** we add observability — tracing every tool call, measuring latency, and inspecting what the agent decided to do and why. That's where this moves from a demo to something you can operate in production.

</div>